# Get the data and build the datasets

### Update FastF1

In [ ]:
# %pip install -U fastf1 pyarrow tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\felip\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


### Imports and configuration

In [ ]:
from pathlib import Path

import pandas as pd
import fastf1
from fastf1.ergast import Ergast
from tqdm.auto import tqdm


START_SEASON = 2018
END_SEASON = 2025

OUTPUT_DIR = Path(".")
CACHE_DIR = Path("fastf1_cache")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

fastf1.Cache.enable_cache(str(CACHE_DIR))
fastf1.set_log_level("WARNING")

# limit=1000 allows a complete season of driver results
# to be returned in one paginated response.
ergast = Ergast(
    result_type="pandas",
    auto_cast=True,
    limit=1000
)

print("FastF1 version:", fastf1.__version__)

FastF1 version: 3.8.3


In [23]:
import time


def iterate_all_pages(response, pause_seconds=1.0):
    """
    Yield every available FastF1/Jolpica result page.

    'No more data after this response' means pagination
    finished successfully.
    """
    while True:
        yield response

        try:
            time.sleep(pause_seconds)
            response = response.get_next_result_page()

        except ValueError as error:
            if "No more data after this response" in str(error):
                break
            raise

### Cleaning functions

In [24]:
def clean_string(series: pd.Series) -> pd.Series:
    """Clean text columns while preserving missing values."""
    return (
        series.astype("string")
        .str.strip()
        .replace("", pd.NA)
    )


def nullable_integer(series: pd.Series) -> pd.Series:
    """Convert numeric columns to nullable integers."""
    return pd.to_numeric(
        series,
        errors="coerce"
    ).astype("Int64")


def parse_qualifying_time(series: pd.Series) -> pd.Series:
    """
    Convert qualifying times such as 1:23.456 into timedeltas.
    Missing Q2 and Q3 times remain NaT.
    """
    if pd.api.types.is_timedelta64_dtype(series):
        return series

    values = (
        series.astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

    # Convert mm:ss.sss into hh:mm:ss.sss.
    one_colon = values.str.count(":").eq(1)
    values = values.mask(
        one_colon,
        "00:" + values
    )

    return pd.to_timedelta(
        values,
        errors="coerce"
    )


def clean_event_date(value) -> pd.Timestamp:
    """Return a timezone-naive event date."""
    timestamp = pd.Timestamp(value)

    if pd.isna(timestamp):
        return pd.NaT

    if timestamp.tzinfo is not None:
        timestamp = timestamp.tz_localize(None)

    return timestamp.normalize()

### Retrieve all seasons

In [25]:
qualifying_frames = []
race_frames = []
failures = []

for season in tqdm(
    range(START_SEASON, END_SEASON + 1),
    desc="Downloading seasons"
):
    # ============================================================
    # EVENT SCHEDULE
    # ============================================================
    try:
        schedule = fastf1.get_event_schedule(
            season,
            include_testing=False,
            backend="fastf1"
        ).copy()

        schedule["RoundNumber"] = pd.to_numeric(
            schedule["RoundNumber"],
            errors="coerce"
        )

        schedule = schedule.loc[
            schedule["RoundNumber"] > 0
        ].copy()

        schedule["RoundNumber"] = (
            schedule["RoundNumber"].astype(int)
        )

        schedule_by_round = schedule.set_index(
            "RoundNumber"
        )

    except Exception as error:
        failures.append({
            "season": season,
            "round": pd.NA,
            "session": "schedule",
            "error": f"{type(error).__name__}: {error}"
        })
        continue

    # ============================================================
    # ALL QUALIFYING RESULTS FOR THE SEASON
    # ============================================================
    try:
        qualifying_response = ergast.get_qualifying_results(
            season=season,
            limit=100
        )

        for qualifying_page in iterate_all_pages(
            qualifying_response
        ):
            for metadata, qualifying_raw in zip(
                qualifying_page.description.to_dict("records"),
                qualifying_page.content
            ):
                qualifying_raw = pd.DataFrame(
                    qualifying_raw
                ).copy()

                if qualifying_raw.empty:
                    continue

                round_number = int(metadata["round"])

                event = schedule_by_round.loc[round_number]

                for column in ["Q1", "Q2", "Q3"]:
                    if column not in qualifying_raw.columns:
                        qualifying_raw[column] = pd.NA

                qualifying_event = pd.DataFrame({
                    "season": season,
                    "round": round_number,
                    "event_name": str(event["EventName"]),
                    "location": str(event["Location"]),
                    "event_date": clean_event_date(
                        event["EventDate"]
                    ),
                    "event_format": str(event["EventFormat"]),
                    "DriverId": clean_string(
                        qualifying_raw["driverId"]
                    ),
                    "TeamId": clean_string(
                        qualifying_raw["constructorId"]
                    ),
                    "qualifying_position": nullable_integer(
                        qualifying_raw["position"]
                    ),
                    "Q1": parse_qualifying_time(
                        qualifying_raw["Q1"]
                    ),
                    "Q2": parse_qualifying_time(
                        qualifying_raw["Q2"]
                    ),
                    "Q3": parse_qualifying_time(
                        qualifying_raw["Q3"]
                    )
                })

                qualifying_frames.append(qualifying_event)

    except Exception as error:
        failures.append({
            "season": season,
            "round": pd.NA,
            "session": "Q-season",
            "error": f"{type(error).__name__}: {error}"
        })

    # ============================================================
    # ALL RACE RESULTS FOR THE SEASON
    # ============================================================
    try:
        race_response = ergast.get_race_results(
            season=season,
            limit=100
        )

        for race_page in iterate_all_pages(race_response):
            for metadata, race_raw in zip(
                race_page.description.to_dict("records"),
                race_page.content
            ):
                race_raw = pd.DataFrame(race_raw).copy()

                if race_raw.empty:
                    continue

                round_number = int(metadata["round"])

                race_event = pd.DataFrame({
                    "season": season,
                    "round": round_number,
                    "DriverId": clean_string(
                        race_raw["driverId"]
                    ),
                    "TeamId": clean_string(
                        race_raw["constructorId"]
                    ),
                    "grid_position": nullable_integer(
                        race_raw["grid"]
                    ),
                    "race_finishing_position": nullable_integer(
                        race_raw["position"]
                    ),
                    "classified_position": clean_string(
                        race_raw["positionText"]
                    ),
                    "status": clean_string(
                        race_raw["status"]
                    ),
                    "raw_points": pd.to_numeric(
                        race_raw["points"],
                        errors="coerce"
                    ),
                    "laps_completed": nullable_integer(
                        race_raw["laps"]
                    )
                })

                race_frames.append(race_event)
            
    except Exception as error:
        failures.append({
            "season": season,
            "round": pd.NA,
            "session": "R-season",
            "error": f"{type(error).__name__}: {error}"
        })

### Check for failures before continuing

In [26]:
failures_df = pd.DataFrame(failures)

pd.set_option("display.max_colwidth", None)

if failures_df.empty:
    print("All seasons downloaded successfully.")
else:
    display(
        failures_df.sort_values(
            ["season", "round", "session"]
        )
    )

All seasons downloaded successfully.


### Combine the results

In [27]:
if not qualifying_frames:
    raise RuntimeError(
        "No qualifying results were collected."
    )

if not race_frames:
    raise RuntimeError(
        "No race results were collected."
    )

qualifying_results = pd.concat(
    qualifying_frames,
    ignore_index=True
)

race_results = pd.concat(
    race_frames,
    ignore_index=True
)

### Enforce the exact schemas

In [28]:
QUALIFYING_COLUMNS = [
    "season",
    "round",
    "event_name",
    "location",
    "event_date",
    "event_format",
    "DriverId",
    "TeamId",
    "qualifying_position",
    "Q1",
    "Q2",
    "Q3"
]

RACE_COLUMNS = [
    "season",
    "round",
    "DriverId",
    "TeamId",
    "grid_position",
    "race_finishing_position",
    "classified_position",
    "status",
    "raw_points",
    "laps_completed"
]

qualifying_results = qualifying_results[
    QUALIFYING_COLUMNS
]

race_results = race_results[
    RACE_COLUMNS
]

### Check Coverage

In [29]:
qualifying_coverage = (
    qualifying_results
    .groupby("season")
    .agg(
        rounds=("round", "nunique"),
        rows=("DriverId", "size"),
        drivers=("DriverId", "nunique")
    )
)

race_coverage = (
    race_results
    .groupby("season")
    .agg(
        rounds=("round", "nunique"),
        rows=("DriverId", "size"),
        drivers=("DriverId", "nunique")
    )
)

print("QUALIFYING COVERAGE")
display(qualifying_coverage)

print("RACE COVERAGE")
display(race_coverage)

QUALIFYING COVERAGE


,rounds,rows,drivers
season,,,
2018,21,420,20
2019,21,418,20
2020,17,340,23
2021,22,439,21
2022,22,440,22
2023,22,440,22
2024,24,479,24
2025,24,479,21


RACE COVERAGE


,rounds,rows,drivers
season,,,
2018,21,420,20
2019,21,420,20
2020,17,340,23
2021,22,440,21
2022,22,440,22
2023,22,440,22
2024,24,479,24
2025,24,479,21


In [33]:
expected_rounds = {
    2018: 21,
    2019: 21,
    2020: 17,
    2021: 22,
    2022: 22,
    2023: 22,
    2024: 24,
    2025: 24
}

actual_q_rounds = (
    qualifying_results.groupby("season")["round"].nunique()
)

actual_r_rounds = (
    race_results.groupby("season")["round"].nunique()
)

for season, expected in expected_rounds.items():
    assert actual_q_rounds.loc[season] == expected, (
        f"Qualifying incomplete for {season}"
    )

    assert actual_r_rounds.loc[season] == expected, (
        f"Race results incomplete for {season}"
    )

print("All seasons have complete round coverage.")

All seasons have complete round coverage.


### Review Datasets

In [35]:
qualifying_results

,season,round,event_name,location,event_date,event_format,DriverId,TeamId,qualifying_position,Q1,Q2,Q3
0,2018,1,Australian Grand Prix,Melbourne,2018-03-25,conventional,hamilton,mercedes,1,0 days 00:01:22.824000,0 days 00:01:22.051000,0 days 00:01:21.164000
1,2018,1,Australian Grand Prix,Melbourne,2018-03-25,conventional,raikkonen,ferrari,2,0 days 00:01:23.096000,0 days 00:01:22.507000,0 days 00:01:21.828000
2,2018,1,Australian Grand Prix,Melbourne,2018-03-25,conventional,vettel,ferrari,3,0 days 00:01:23.348000,0 days 00:01:21.944000,0 days 00:01:21.838000
3,2018,1,Australian Grand Prix,Melbourne,2018-03-25,conventional,max_verstappen,red_bull,4,0 days 00:01:23.483000,0 days 00:01:22.416000,0 days 00:01:21.879000
4,2018,1,Australian Grand Prix,Melbourne,2018-03-25,conventional,ricciardo,red_bull,5,0 days 00:01:23.494000,0 days 00:01:22.897000,0 days 00:01:22.152000
...,...,...,...,...,...,...,...,...,...,...,...,...
3450,2025,24,Abu Dhabi Grand Prix,Yas Island,2025-12-07,conventional,hamilton,ferrari,16,0 days 00:01:23.394000,NaT,NaT
3451,2025,24,Abu Dhabi Grand Prix,Yas Island,2025-12-07,conventional,albon,williams,17,0 days 00:01:23.416000,NaT,NaT
3452,2025,24,Abu Dhabi Grand Prix,Yas Island,2025-12-07,conventional,hulkenberg,sauber,18,0 days 00:01:23.450000,NaT,NaT
3453,2025,24,Abu Dhabi Grand Prix,Yas Island,2025-12-07,conventional,gasly,alpine,19,0 days 00:01:23.468000,NaT,NaT


In [36]:
race_results

,season,round,DriverId,TeamId,grid_position,race_finishing_position,classified_position,status,raw_points,laps_completed
0,2018,1,vettel,ferrari,3,1,1,Finished,25.0,58
1,2018,1,hamilton,mercedes,1,2,2,Finished,18.0,58
2,2018,1,raikkonen,ferrari,2,3,3,Finished,15.0,58
3,2018,1,ricciardo,red_bull,8,4,4,Finished,12.0,58
4,2018,1,alonso,mclaren,10,5,5,Finished,10.0,58
...,...,...,...,...,...,...,...,...,...,...
3453,2025,24,albon,williams,17,16,16,Finished,0.0,58
3454,2025,24,hadjar,rb,9,17,17,Lapped,0.0,57
3455,2025,24,lawson,rb,13,18,18,Lapped,0.0,57
3456,2025,24,gasly,alpine,19,19,19,Lapped,0.0,57


### Store datasets

In [37]:
qualifying_results.to_parquet(
    "qualifying_results.parquet",
    index=False
)

race_results.to_parquet(
    "race_results.parquet",
    index=False
)